# CineMatch: Historical Movie Performance Analysis

This notebook explores movie metadata and trains a **retrospective** classification baseline. The objective is to demonstrate a defensible analytics workflow—not to claim live box-office prediction.

The baseline deliberately excludes obvious post-release fields such as final revenue, popularity, vote counts, and vote average from its feature set. Revenue is used only to define the historical outcome label.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from cinematch.analytics import prepare_historical_performance_data, train_historical_baseline
from cinematch.config import CREDITS_FILE, MOVIES_FILE
from cinematch.data import load_tmdb_data

if not MOVIES_FILE.exists() or not CREDITS_FILE.exists():
    raise FileNotFoundError("Add tmdb_5000_movies.csv and tmdb_5000_credits.csv to the data/ folder before running this notebook.")


## Load data

The analysis begins with the movie metadata table. Credits are also loaded for consistency with the project data pipeline, though the baseline below uses only selected movie-level fields.


In [ ]:
movies = load_tmdb_data(str(MOVIES_FILE), str(CREDITS_FILE))
movies.shape


## Exploratory view

Budget and revenue are shown on logarithmic axes because movie financial values are strongly right-skewed. Records with zero or unavailable financial values are excluded only from this visualization.


In [ ]:
financial = movies[(movies["budget"] > 0) & (movies["revenue"] > 0)].copy()
plt.figure(figsize=(9, 6))
sns.scatterplot(data=financial, x="budget", y="revenue", hue="vote_average", palette="viridis", alpha=0.55, legend=False)
plt.xscale("log")
plt.yscale("log")
plt.title("Historical Budget and Revenue Distribution")
plt.xlabel("Budget (log scale)")
plt.ylabel("Revenue (log scale)")
plt.tight_layout()
plt.show()


## Prepare a leakage-aware baseline

The label is `profitable`, defined here as revenue greater than or equal to budget. The input feature set uses budget, runtime, release year, genre/keyword counts, primary genre, and original language. This design is still a simplification and should not be treated as a production investment model.


In [ ]:
X, y = prepare_historical_performance_data(movies)
print("Feature matrix:", X.shape)
print("Outcome distribution:")
print(y.value_counts(normalize=True).rename("share"))
X.head()


## Train and evaluate

A logistic-regression pipeline handles missing values, scaling, categorical encoding, and hyperparameter selection with five-fold cross-validation.


In [ ]:
baseline_model, metrics = train_historical_baseline(movies, random_state=42)
pd.Series(metrics, name="value")


## Interpretation

Metrics from this notebook describe how well the selected historical metadata separates the chosen retrospective outcome. They should not be interpreted as a claim that a new movie's commercial performance can be known before release. Real pre-release forecasting requires carefully timestamped features, more complete production data, external validation, and explicit uncertainty analysis.
